In [23]:
# Import libraries
import pandas as pd
import numpy as np
import requests
import os

print("Libraries loaded")

Libraries loaded


In [24]:
# Where we save our data
RAW = "data/raw"
PROCESSED = "data/processed"

os.makedirs(RAW, exist_ok=True)
os.makedirs(PROCESSED, exist_ok=True)

print("Folders ready")

Folders ready


In [25]:
# CMS API Link
BASE_URL = "https://data.cms.gov/data-api/v1/dataset/b101b457-ffa4-49bb-8fd9-27c1266086e2/data"

# The five GLP-1 drugs we want
GLP1_DRUGS = ['semaglutide', 'tirzepatide', 'dulaglutide', 
               'liraglutide', 'exenatide']

all_data = []

for drug in GLP1_DRUGS:
    
    params = {
        'filter[gnrc_name]': drug,
        'size': 5000
    }
    
    response = requests.get(BASE_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        all_data.extend(data)
        print(f"Got {len(data)} rows for {drug}")
    else:
        print(f"Failed for {drug}: {response.status_code}")

print(f"\nTotal rows collected: {len(all_data)}")

Got 5000 rows for semaglutide
Got 5000 rows for tirzepatide
Got 5000 rows for dulaglutide
Got 5000 rows for liraglutide
Got 5000 rows for exenatide

Total rows collected: 25000


In [26]:
df = pd.DataFrame(all_data)

# Shape of data
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print()

# Column names
print("Columns we have:")
print(df.columns.tolist())

Rows: 25000
Columns: 22

Columns we have:
['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_Type', 'Prscrbr_Type_Src', 'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']


In [27]:
# Keep only the columns we need
columns_we_need = [
    'Prscrbr_NPI',
    'Prscrbr_Last_Org_Name', 
    'Prscrbr_First_Name',
    'Prscrbr_City',
    'Prscrbr_State_Abrvtn',
    'Prscrbr_Type',
    'Brnd_Name',
    'Gnrc_Name',
    'Tot_Clms',
    'Tot_Drug_Cst',
    'Tot_Benes'
]

df = df[columns_we_need]

print(f"Columns now: {df.shape[1]}")
print()

print(df.head())

Columns now: 11

  Prscrbr_NPI Prscrbr_Last_Org_Name Prscrbr_First_Name Prscrbr_City  \
0  1003000126             Enkeshafi            Ardalan     Bethesda   
1  1003000126             Enkeshafi            Ardalan     Bethesda   
2  1003000126             Enkeshafi            Ardalan     Bethesda   
3  1003000126             Enkeshafi            Ardalan     Bethesda   
4  1003000126             Enkeshafi            Ardalan     Bethesda   

  Prscrbr_State_Abrvtn       Prscrbr_Type             Brnd_Name  \
0                   MD  Internal Medicine   Amlodipine Besylate   
1                   MD  Internal Medicine  Atorvastatin Calcium   
2                   MD  Internal Medicine               Eliquis   
3                   MD  Internal Medicine  Escitalopram Oxalate   
4                   MD  Internal Medicine   Hydrochlorothiazide   

              Gnrc_Name Tot_Clms Tot_Drug_Cst Tot_Benes  
0   Amlodipine Besylate       19       246.17        14  
1  Atorvastatin Calcium       11     

In [28]:
# Rename columns 
df = df.rename(columns={
    'Prscrbr_NPI': 'npi',
    'Prscrbr_Last_Org_Name': 'last_name',
    'Prscrbr_First_Name': 'first_name',
    'Prscrbr_City': 'city',
    'Prscrbr_State_Abrvtn': 'state',
    'Prscrbr_Type': 'specialty',
    'Brnd_Name': 'brand_name',
    'Gnrc_Name': 'generic_name',
    'Tot_Clms': 'total_claims',
    'Tot_Drug_Cst': 'total_cost',
    'Tot_Benes': 'total_patients'
})

# Keep only GLP-1 drugs
glp1_list = ['semaglutide', 'tirzepatide', 'dulaglutide', 
             'liraglutide', 'exenatide']

df = df[df['generic_name'].str.lower().isin(glp1_list)]

print(f"Rows after filtering to GLP-1 only: {len(df)}")
print()
print(df.head())

Rows after filtering to GLP-1 only: 220

             npi   last_name first_name            city state  \
475   1003002049  Srinivasan    Lakshmi         Fremont    CA   
481   1003002049  Srinivasan    Lakshmi         Fremont    CA   
1014  1003004201   Philbrick    Natalie  Corpus Christi    TX   
1027  1003004201   Philbrick    Natalie  Corpus Christi    TX   
1047  1003004201   Philbrick    Natalie  Corpus Christi    TX   

              specialty brand_name generic_name total_claims total_cost  \
475       Endocrinology    Ozempic  Semaglutide           70  133545.13   
481       Endocrinology  Trulicity  Dulaglutide           15   32458.42   
1014  Internal Medicine    Ozempic  Semaglutide           66   70837.96   
1027  Internal Medicine   Rybelsus  Semaglutide           16   16173.58   
1047  Internal Medicine  Trulicity  Dulaglutide           44   59138.23   

     total_patients  
475              18  
481                  
1014             14  
1027                 
1047   

In [29]:
# Fix generic_name to all lowercase
df['generic_name'] = df['generic_name'].str.lower()

# Fix total_claims and total_cost to be numbers not text
df['total_claims'] = pd.to_numeric(df['total_claims'], errors='coerce')
df['total_cost'] = pd.to_numeric(df['total_cost'], errors='coerce')
df['total_patients'] = pd.to_numeric(df['total_patients'], errors='coerce')

# Fill missing values with 0
df['total_patients'] = df['total_patients'].fillna(0)
df['total_claims'] = df['total_claims'].fillna(0)

# Check for any remaining missing values
print("Missing values per column:")
print(df.isnull().sum())
print()
print(f"Final clean dataset: {len(df)} rows")

Missing values per column:
npi               0
last_name         0
first_name        0
city              0
state             0
specialty         0
brand_name        0
generic_name      0
total_claims      0
total_cost        0
total_patients    0
dtype: int64

Final clean dataset: 220 rows
